---
title: "Chapter 3: Data Organization and Discovery Results"
short_title: "3. Data Organization and Discovery"
---

## 1. Structure of the working corpus

Each ID range (one per member) has its own data folder (`--data-dir`) on the disk of the Colab runtime (`/content/g11/<name>_<first ID>_<last ID>`), outside the code repository. After each crawl step, the changed files are packed into a `.tar.gz` snapshot saved to Google Drive; a new runtime restores the folder from these snapshots and continues. Inside the folder, each paper has its own sub-folder named `<yymm-id>` (for example `2604-05535` for the ID `2604.05535`), following the convention suggested by the statement:

```text
<data-dir>/
├── work/
│   └── paper_manifest.jsonl     one line per ID: discovery status, version list
├── corpus/<yymm-id>/
│   ├── metadata.json            metadata of the paper and of each version
│   ├── versions.json            download/extraction status of each version
│   ├── v1/, v2/, ...            .tex/.bib/.bbl files, keeping the sub-folders of the source archive
│   ├── references_all.json      every reference returned by Semantic Scholar
│   └── references.json          only the references with an arXiv ID
└── logs/                        CSV logs used for measurement and reporting
```

A PDF-only version or a version without TeX still has an (empty) `v<N>/` folder and a status record in `versions.json`, so no paper is dropped from the corpus.

## 2. File contents

| File | Main fields |
| :--- | :--- |
| `metadata.json` | `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue`, `primary_category`, `categories`, `versions[]` (per-version metadata) |
| `versions.json` | Per version (`kept_bytes` and `kept_files` include `.bbl` files): `status`, `source_format`, `downloaded_bytes`, `original_bytes`, `image_bytes`, `kept_bytes`, `kept_files`, `attempts`, `finished_at` |
| `references_all.json` | `status`, `s2_paper_id`, `s2_reference_count`, `arxiv_dates_complete`, `references[]` with `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_s2_id`, `metadata_status`, … |
| `references.json` | Keyed by the cited arXiv ID (`yymm-id` form): `title`, `authors`, `submission_date`, `semantic_scholar_id` |

## 3. Mapping to the tables of the final dataset

The statement specifies two tables to be published in Milestone 3. The working corpus already holds all the data needed to build them without crawling again:

| Table | Columns | Source |
| :--- | :--- | :--- |
| `papers` | `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue` | The fields of the same name at the top level of `metadata.json` |
| `references` | `arxiv_id`, `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_s2_id` | The references with a `ref_arxiv_id` in `references_all.json` |

## 4. Status codes

| Stage | Status | Meaning |
| :--- | :--- | :--- |
| Discovery | `success` / `not_found` / `api_error` | The ID exists / arXiv has no such ID / API error, retried in a later pass |
| Version | `success` / `pdf_only` / `no_tex` / `not_found` / `failed` | At least one `.tex`/`.bib` file / PDF only / no `.tex`/`.bib` file (DOCX, PostScript, archive with images only) / arXiv answers 404 (withdrawn version, [Chapter 6](./06_errors_limitations.ipynb)) / error, retried in a later pass |
| References | `success` / `empty` / `unavailable` / `not_found` / `api_error` | See [Chapter 2](./02_methodology.ipynb), Section 6 |

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

## 5. ID range of each member

The number of IDs is read from the manifest of each data folder.

In [ ]:
m = members()
ranges = pd.DataFrame({
    "Member": m["member"], "First ID": m["first_id"], "Last ID": m["last_id"],
    "IDs": m["ids"], "Existing papers": m["papers"], "Not found": m["ids_not_found"],
    "API errors": m["ids_api_error"], "Versions": m["versions"]})
g = group()
ranges.loc[len(ranges)] = ["Group", g["first_id"], g["last_id"], g["ids"], g["papers"],
                           g["ids_not_found"], g["ids_api_error"], g["versions"]]
table(ranges, "Table 3.1. ID ranges and discovery results per member")

## 6. Number of versions per paper

The statement requires **every version** to be crawled; the distribution below shows how many papers have more than one version.

In [ ]:
papers = load("papers")
found = papers[papers["discovery_status"] == "success"]
dist = found.groupby("n_versions").size().reset_index(name="papers")
dist["share"] = dist["papers"] / dist["papers"].sum()
fig = px.bar(dist, x="n_versions", y="papers", text=dist["share"].map(pct),
             color_discrete_sequence=COLORS,
             title=f"Versions per paper ({len(found):,} existing papers)",
             labels={"n_versions": "Versions", "papers": "Papers"})
fig.update_xaxes(dtick=1)
show(fig)
print(f"Mean of {found['n_versions'].mean():.3f} versions per paper; "
      f"{(found['n_versions'] > 1).mean():.2%} of the papers have two or more versions.")

## 7. Subject areas

The primary category (`primary_category`) is taken from the arXiv metadata.

In [ ]:
cats = found["primary_category"].fillna("(none)")
top = cats.value_counts().head(20).rename_axis("category").reset_index(name="papers")
fig = px.bar(top.iloc[::-1], x="papers", y="category", orientation="h", color_discrete_sequence=COLORS,
             title="Top 20 primary categories by number of papers",
             labels={"papers": "Papers", "category": "primary_category"})
fig.update_layout(height=600)
show(fig)
archive = cats.str.split(".").str[0].value_counts().rename_axis("archive").reset_index(name="papers")
fig = px.pie(archive, names="archive", values="papers", color_discrete_sequence=COLORS,
             title="Share of papers by archive (the part of primary_category before the dot)")
show(fig)